# Tiny LLM — SFT Experiment Matrix

Systematic sweep over **batch size, learning rate, epochs, warmup ratio, and data mix** to find the best SFT checkpoint for our 40M-parameter model.

**Key hypotheses** (from [arxiv 2412.13337](https://arxiv.org/abs/2412.13337) + VL07/VL09):
1. Larger effective batch + lower LR → better benchmarks for small LLMs
2. More epochs may help (model is undertrained at 2-3 epochs)
3. MC-heavy data mix → direct benchmark improvement
4. Warmup ratio matters — shorter warmup may help with small datasets
5. Constrained decoding for MC evals is a must (VL09)

**Run time budget:** ~1 hour per experiment on a T4/A100. The full matrix takes 6-12 hours.

---

In [ ]:
import os
try:
    os.getcwd()
except OSError:
    os.chdir('/workspace' if os.path.exists('/workspace') else os.path.expanduser('~'))
print(f'Working directory: {os.getcwd()}')

## 1. Environment Setup

In [ ]:
import pathlib, os

# --- Auto-detect environment ---
ON_COLAB = os.path.exists('/content') and 'COLAB_RELEASE_TAG' in os.environ
ON_RUNPOD = os.path.exists('/workspace') or 'RUNPOD_POD_ID' in os.environ

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/tiny_llm/sft_experiment_matrix')
    WORK_DIR = pathlib.Path('/content')
elif ON_RUNPOD:
    DRIVE_ROOT = pathlib.Path('/workspace/sft_experiment_matrix')
    WORK_DIR = pathlib.Path('/workspace')
else:
    DRIVE_ROOT = pathlib.Path.home() / 'sft_experiment_matrix'
    WORK_DIR = pathlib.Path.home()

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Environment: {"Colab" if ON_COLAB else "RunPod" if ON_RUNPOD else "Local"}'  )
print(f'Results dir: {DRIVE_ROOT}')
print(f'Work dir:    {WORK_DIR}')

In [ ]:
import os, subprocess

REPO_OWNER = 'marcbloech'
REPO_NAME = '40m-tiny-llm'
BRANCH = 'post-training'
REPO_DIR = str(WORK_DIR / REPO_NAME)

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_colab_secret(name):
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

GH_TOKEN = (
    os.environ.get('GH_TOKEN')
    or os.environ.get('GITHUB_TOKEN')
    or get_colab_secret('GH_TOKEN')
    or get_colab_secret('GITHUB_TOKEN')
)
if not GH_TOKEN:
    raise RuntimeError(
        'No GitHub token found. Set GH_TOKEN env var or add as Colab Secret.\n'
        'On RunPod: export GH_TOKEN=ghp_... before launching the notebook.'
    )

if os.path.exists(REPO_DIR):
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)

clone_url = f'https://{GH_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
public_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', clone_url, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', public_url], check=True)
del GH_TOKEN, clone_url

os.chdir(REPO_DIR)
print(f'Cloned into {REPO_DIR}')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF token (or Enter to skip): ').strip() or None
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    print(f'HF token set (len={len(HF_TOKEN)})')
else:
    print('No HF token — public datasets will still work.')

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ['PATH']
!uv --version
!cd {REPO_DIR} && uv sync --extra all 2>&1 | tail -10

## 2. GPU Detection & Auto-Tune

In [ ]:
import subprocess, json

probe = subprocess.run(
    ['uv', 'run', 'python', '-c',
     'import json, torch;'
     " d = {'cuda': torch.cuda.is_available()};"
     " d['name'] = torch.cuda.get_device_name(0) if d['cuda'] else '';"
     " d['cap']  = list(torch.cuda.get_device_capability(0)) if d['cuda'] else [0,0];"
     " d['vram_gb'] = round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if d['cuda'] else 0;"
     ' print(json.dumps(d))'],
    cwd=REPO_DIR, capture_output=True, text=True, check=True,
)
gpu = json.loads(probe.stdout.strip().splitlines()[-1])
print('GPU probe:', gpu)

if not gpu['cuda']:
    raise RuntimeError('No CUDA GPU — switch runtime to GPU.')

name = gpu.get('name', '')
sm = gpu['cap'][0] if gpu['cap'] else 0
vram = gpu.get('vram_gb', 0)
ampere_or_newer = sm >= 8

AMP_DTYPE = 'bf16' if ampere_or_newer else 'fp16'
NUM_WORKERS = 4 if ampere_or_newer else 2

if vram >= 40:
    MAX_BATCH = 128
elif vram >= 20:
    MAX_BATCH = 96
elif vram >= 14:
    MAX_BATCH = 64
else:
    MAX_BATCH = 32

print(f'GPU: {name} ({vram} GB VRAM, sm={sm})')
print(f'AMP: {AMP_DTYPE}, max micro-batch: {MAX_BATCH}, workers: {NUM_WORKERS}')

## 3. Stage Base Checkpoint + Pretrain Val

In [ ]:
import pathlib, shutil, urllib.request

REPO = pathlib.Path(REPO_DIR)

# --- Base checkpoint ---
# On Colab: copy from Drive. On RunPod: expect it in /workspace/ or download.
CKPT_CANDIDATES = [
    pathlib.Path('/content/drive/MyDrive/tiny_llm/shapeC.pt'),  # Colab Drive
    pathlib.Path('/workspace/shapeC.pt'),                        # RunPod workspace
    REPO / 'checkpoints' / 'shapeC.pt',                         # Already in repo
]

LOCAL_CKPT = REPO / 'checkpoints' / 'shapeC.pt'
LOCAL_CKPT.parent.mkdir(parents=True, exist_ok=True)

if not LOCAL_CKPT.exists():
    for candidate in CKPT_CANDIDATES:
        if candidate.exists() and candidate != LOCAL_CKPT:
            shutil.copy2(candidate, LOCAL_CKPT)
            print(f'Checkpoint copied from {candidate}')
            break
    else:
        raise FileNotFoundError(
            f'shapeC.pt not found. Place it at one of:\n'
            + '\n'.join(f'  {c}' for c in CKPT_CANDIDATES)
        )

size_mb = LOCAL_CKPT.stat().st_size / 1e6
print(f'Checkpoint staged: {LOCAL_CKPT} ({size_mb:.1f} MB)')

# --- Pretrain val.bin for catastrophic forgetting probe ---
FORGETTING_OVERRIDES = []
dst = REPO / 'data' / 'tokenized' / 'val.bin'
dst.parent.mkdir(parents=True, exist_ok=True)
if not dst.exists():
    HF_VAL_BIN_URLS = [
        'https://huggingface.co/datasets/marcbloech/tiny-llm-tokenised/resolve/main/data/val.bin',
        'https://huggingface.co/datasets/marcbloech/tiny-llm-tokenised/resolve/main/data/tokenized/val.bin',
    ]
    for url in HF_VAL_BIN_URLS:
        try:
            urllib.request.urlretrieve(url, dst)
            print(f'val.bin restored from {url} ({dst.stat().st_size/1e6:.1f} MB)')
            break
        except Exception:
            pass
    else:
        print('No pretrain val.bin — disabling forgetting probe.')
        FORGETTING_OVERRIDES = ['posttraining.pretrain_val_path=']
else:
    print(f'val.bin already present ({dst.stat().st_size/1e6:.1f} MB)')

## 4. Define the Experiment Matrix

We sweep over **5 axes**:
- **Effective batch size**: 32, 64, 128, 256
- **Learning rate**: 5e-6, 1e-5, 2e-5
- **Epochs**: 1, 4, 5, 8, 10, 20, 50
- **Warmup ratio**: 0.03, 0.05, 0.10
- **MC data weight**: normal (~34%), heavy (~65%)

Running the full cross-product would be thousands of runs. Instead we use a **structured sweep**: vary one axis at a time from a strong baseline, then combine the winners.

In [ ]:
import datetime

CKPT_PATH = 'checkpoints/shapeC.pt'
CONFIG = 'configs/sft.toml'
RUN_STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Dataset cache — reuse across runs to avoid re-downloading
if ON_COLAB:
    SFT_CACHE_DIR = pathlib.Path('/content/drive/MyDrive/tiny_llm/sft_dataset_analysis/sft')
else:
    SFT_CACHE_DIR = DRIVE_ROOT / 'sft_cache'
SFT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

APPROX_TRAIN_SIZE = 50_000

# --- Current best Optuna weights (Trial 0) ---
BASELINE_WEIGHTS = {
    'alpaca': 0.5, 'oasst1': 2.0, 'dolly': 1.2,
    'arc': 2.0, 'openbookqa': 2.0, 'sciq': 1.0,
    'winogrande': 0.5, 'hellaswag': 0.3, 'lambada_like': 1.0,
}

# MC-heavy weights: boost MC sources, reduce chat/instruction sources
MC_HEAVY_WEIGHTS = {
    'alpaca': 0.25, 'oasst1': 0.5, 'dolly': 0.3,
    'arc': 4.0, 'openbookqa': 4.0, 'sciq': 2.0,
    'winogrande': 2.0, 'hellaswag': 2.0, 'lambada_like': 0.5,
}


def make_experiment(name, *, batch_size=16, grad_accum=2, lr=2e-5, epochs=3,
                    warmup_ratio=0.10, weights=None, extra_overrides=None):
    if weights is None:
        weights = BASELINE_WEIGHTS
    eff_batch = batch_size * grad_accum
    steps_per_epoch = max(1, APPROX_TRAIN_SIZE // eff_batch)
    total_steps = steps_per_epoch * epochs
    return {
        'name': name,
        'batch_size': batch_size,
        'grad_accum': grad_accum,
        'effective_batch': eff_batch,
        'lr': lr,
        'epochs': epochs,
        'warmup_ratio': warmup_ratio,
        'weights': weights,
        'extra_overrides': extra_overrides or [],
        'est_steps_per_epoch': steps_per_epoch,
        'est_total_steps': total_steps,
    }


def fit_batch(target_eff, max_micro=MAX_BATCH):
    bs = min(target_eff, max_micro)
    accum = max(1, target_eff // bs)
    return bs, accum


# =====================================================================
# PHASE 1: Batch Size + LR sweep
# =====================================================================
phase1 = []
for eff_batch in [32, 64, 128, 256]:
    bs, accum = fit_batch(eff_batch)
    for lr in [5e-6, 1e-5, 2e-5, 5e-5, 1e-4]:
        phase1.append(make_experiment(
            f'p1_eb{eff_batch}_lr{lr:g}',
            batch_size=bs, grad_accum=accum, lr=lr,
            epochs=3, warmup_ratio=0.10,
        ))

# =====================================================================
# PHASE 2: Epoch sweep
# =====================================================================
P2_EFF_BATCH = 128
P2_LR = 1e-5

phase2 = []
for ep in [1, 4, 5, 8, 10, 20, 50]:
    bs, accum = fit_batch(P2_EFF_BATCH)
    phase2.append(make_experiment(
        f'p2_ep{ep}',
        batch_size=bs, grad_accum=accum, lr=P2_LR,
        epochs=ep, warmup_ratio=0.10,
    ))

# =====================================================================
# PHASE 3: Warmup ratio sweep
# =====================================================================
P3_EPOCHS = 5

phase3 = []
for warmup in [0.01, 0.03, 0.05, 0.10, 0.15, 0.20]:
    bs, accum = fit_batch(P2_EFF_BATCH)
    phase3.append(make_experiment(
        f'p3_wu{warmup:.2f}',
        batch_size=bs, grad_accum=accum, lr=P2_LR,
        epochs=P3_EPOCHS, warmup_ratio=warmup,
    ))

# =====================================================================
# PHASE 4: MC-heavy data mix
# =====================================================================
P4_WARMUP = 0.05

phase4 = []
bs, accum = fit_batch(P2_EFF_BATCH)
phase4.append(make_experiment(
    'p4_mc_heavy',
    batch_size=bs, grad_accum=accum, lr=P2_LR,
    epochs=P3_EPOCHS, warmup_ratio=P4_WARMUP,
    weights=MC_HEAVY_WEIGHTS,
))
for ep in [8, 10, 20]:
    phase4.append(make_experiment(
        f'p4_mc_heavy_ep{ep}',
        batch_size=bs, grad_accum=accum, lr=P2_LR,
        epochs=ep, warmup_ratio=P4_WARMUP,
        weights=MC_HEAVY_WEIGHTS,
    ))

# =====================================================================
# PHASE 5: Kitchen sink
# =====================================================================
phase5 = []
for eff_batch in [128, 256]:
    bs, accum = fit_batch(eff_batch)
    phase5.append(make_experiment(
        f'p5_kitchen_eb{eff_batch}',
        batch_size=bs, grad_accum=accum, lr=P2_LR,
        epochs=P3_EPOCHS, warmup_ratio=P4_WARMUP,
        weights=MC_HEAVY_WEIGHTS,
    ))

ALL_PHASES = {
    'phase1_batch_lr': phase1,
    'phase2_epochs': phase2,
    'phase3_warmup': phase3,
    'phase4_mc_heavy': phase4,
    'phase5_kitchen_sink': phase5,
}

total = sum(len(v) for v in ALL_PHASES.values())
print(f'Experiment matrix: {total} total runs across {len(ALL_PHASES)} phases')
for phase_name, exps in ALL_PHASES.items():
    print(f'\n  {phase_name}: {len(exps)} runs')
    for exp in exps:
        print(f'    {exp["name"]}: eb={exp["effective_batch"]}, lr={exp["lr"]:g}, '
              f'ep={exp["epochs"]}, wu={exp["warmup_ratio"]:.2f}, '
              f'~{exp["est_total_steps"]} steps')

## 5. Select Which Phase to Run

Set `RUN_PHASE` to the phase you want to run. Run phases sequentially — after each phase, update the `P2_*`, `P3_*`, `P4_*` variables above based on results, then run the next phase.

In [ ]:
# ============= SET THIS =============
RUN_PHASE = 'phase1_batch_lr'  # 'phase1_batch_lr' | 'phase2_epochs' | 'phase3_warmup' | 'phase4_mc_heavy' | 'phase5_kitchen_sink'
# ====================================

experiments = ALL_PHASES[RUN_PHASE]
print(f'Will run {len(experiments)} experiments from {RUN_PHASE}:')
for exp in experiments:
    print(f'  {exp["name"]}: eb={exp["effective_batch"]}, lr={exp["lr"]:g}, '
          f'ep={exp["epochs"]}, wu={exp["warmup_ratio"]:.2f}')

## 6. Run the Experiments

In [ ]:
import os, subprocess, sys, pathlib, json, time, csv, math

REPO = pathlib.Path(REPO_DIR)

env = dict(os.environ,
           NVIDIA_TF32_OVERRIDE='1',
           TORCH_ALLOW_TF32_CUBLAS_OVERRIDE='1',
           PYTHONUNBUFFERED='1')

results = []


def build_overrides(exp):
    steps_per_epoch = exp.get('est_steps_per_epoch', 500)
    eval_interval = max(10, min(100, steps_per_epoch // 3))

    overrides = [
        f'posttraining.amp_dtype={AMP_DTYPE}',
        f'posttraining.batch_size={exp["batch_size"]}',
        f'posttraining.eval_batch_size={exp["batch_size"]}',
        f'posttraining.grad_accum_steps={exp["grad_accum"]}',
        f'posttraining.num_workers={NUM_WORKERS}',
        f'posttraining.learning_rate={exp["lr"]:g}',
        f'posttraining.epochs={exp["epochs"]}',
        f'posttraining.warmup_ratio={exp["warmup_ratio"]:g}',
        f'posttraining.filter_code_and_math=true',
        f'posttraining.checkpoint_dir=checkpoints/sft_matrix/{RUN_PHASE}/{exp["name"]}',
        f'posttraining.dataset_dir={SFT_CACHE_DIR}',
        f'posttraining.eval_interval={eval_interval}',
        f'posttraining.checkpoint_interval={max(eval_interval * 3, 200)}',
        f'posttraining.quick_plot_interval={max(10, eval_interval // 2)}',
        'posttraining.log_interval=10',
        'posttraining.max_checkpoints=2',
        'posttraining.max_best_checkpoints=2',
    ]
    for ds_name, weight in exp['weights'].items():
        overrides.append(f'posttraining.dataset_weights.{ds_name}={weight:g}')
    overrides.extend(FORGETTING_OVERRIDES)
    overrides.extend(exp.get('extra_overrides', []))
    return overrides


def get_best_val_loss(checkpoint_dir):
    metrics_path = pathlib.Path(checkpoint_dir) / 'metrics.csv'
    if not metrics_path.exists():
        return math.inf
    best = math.inf
    with metrics_path.open(newline='') as f:
        for row in csv.DictReader(f):
            raw = row.get('val/loss')
            if raw:
                try:
                    best = min(best, float(raw))
                except ValueError:
                    pass
    return best


def find_best_checkpoint(checkpoint_dir):
    import glob
    d = pathlib.Path(checkpoint_dir)
    best = sorted(glob.glob(str(d / 'best*.pt')))
    steps = sorted(glob.glob(str(d / 'step_*.pt')))
    if best:
        return best[-1]
    if steps:
        return steps[-1]
    return None


print(f'Starting {len(experiments)} experiments...\n')

for i, exp in enumerate(experiments):
    exp_name = exp['name']
    ckpt_dir = REPO / 'checkpoints' / 'sft_matrix' / RUN_PHASE / exp_name
    
    print(f'\n{"="*60}')
    print(f'[{i+1}/{len(experiments)}] {exp_name}')
    print(f'  effective_batch={exp["effective_batch"]}, lr={exp["lr"]:g}, '
          f'epochs={exp["epochs"]}, warmup={exp["warmup_ratio"]:.2f}')
    print(f'  est. {exp["est_total_steps"]} optimizer steps '
          f'({exp["est_steps_per_epoch"]} steps/epoch)')
    print(f'{"="*60}\n')
    
    overrides = build_overrides(exp)
    cmd = [
        'uv', 'run', 'python', 'main.py',
        '--stage', 'posttraining',
        '--config', CONFIG,
        '--checkpoint', CKPT_PATH,
        '--device', 'cuda',
        '--overrides', *overrides,
    ]
    
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(REPO), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True,
    )
    tail = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        tail.append(line)
        if len(tail) > 40:
            tail.pop(0)
    returncode = proc.wait()
    elapsed = time.time() - t0
    
    val_loss = get_best_val_loss(ckpt_dir)
    best_ckpt = find_best_checkpoint(ckpt_dir)
    
    result = {
        'name': exp_name,
        'effective_batch': exp['effective_batch'],
        'lr': exp['lr'],
        'epochs': exp['epochs'],
        'warmup_ratio': exp['warmup_ratio'],
        'mc_heavy': exp['weights'] is MC_HEAVY_WEIGHTS,
        'val_loss': val_loss,
        'best_checkpoint': best_ckpt,
        'elapsed_min': round(elapsed / 60, 1),
        'returncode': returncode,
        'checkpoint_dir': str(ckpt_dir),
    }
    results.append(result)
    
    status = 'OK' if returncode == 0 else f'FAILED (rc={returncode})'
    print(f'\n--- {exp_name}: {status} | val_loss={val_loss:.4f} | {elapsed/60:.1f} min ---\n')

print('\n' + '='*60)
print('ALL EXPERIMENTS COMPLETE')
print('='*60)

## 7. Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df.sort_values('val_loss')
print(f'\n{RUN_PHASE} Results (sorted by val_loss):\n')
print(df[['name', 'effective_batch', 'lr', 'epochs', 'warmup_ratio', 'mc_heavy',
           'val_loss', 'elapsed_min', 'returncode']].to_string(index=False))

best = df.iloc[0]
print(f'\nBest run: {best["name"]} (val_loss={best["val_loss"]:.4f})')
print(f'  -> eb={best["effective_batch"]}, lr={best["lr"]:g}, '
      f'epochs={best["epochs"]}, warmup={best["warmup_ratio"]:.2f}')
if best['best_checkpoint']:
    print(f'  -> checkpoint: {best["best_checkpoint"]}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Val loss by experiment
ax = axes[0]
ax.barh(df['name'], df['val_loss'])
ax.set_xlabel('Val Loss')
ax.set_title(f'{RUN_PHASE}: Val Loss by Experiment')
ax.invert_yaxis()

# Plot 2: Training time
ax = axes[1]
ax.barh(df['name'], df['elapsed_min'], color='orange')
ax.set_xlabel('Time (min)')
ax.set_title(f'{RUN_PHASE}: Training Time')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / f'{RUN_PHASE}_{RUN_STAMP}_results.png'), dpi=150)
plt.show()

## 8. Benchmark Evaluation (ALL Checkpoints)

Run the actual benchmarks (HellaSwag, WinoGrande, OpenBookQA, LAMBADA) on **all successful checkpoints**.

**Why all?** Val-loss is a poor proxy for MC benchmark accuracy (C1 finding from review):
MC targets are 1 token, so they contribute almost nothing to overall val_loss. A model
that memorizes verbose Alpaca answers can dominate val_loss yet fail on MC benchmarks.

In [ ]:
# Evaluate ALL successful runs — val_loss is unreliable for MC benchmark ranking
eval_candidates = [r for r in results
                   if r['returncode'] == 0 and r['best_checkpoint']]

print(f'Evaluating {len(eval_candidates)} checkpoints on benchmarks...')
print('(This takes ~2-5 min per checkpoint)\n')

eval_results = []
for i, r in enumerate(eval_candidates):
    ckpt = r['best_checkpoint']
    print(f'[{i+1}/{len(eval_candidates)}] Evaluating {r["name"]} ({ckpt})...')
    
    eval_cmd = [
        'uv', 'run', 'python', 'main.py',
        '--stage', 'evaluate',
        '--checkpoint', ckpt,
        '--device', 'cuda',
    ]
    
    proc = subprocess.run(
        eval_cmd, cwd=str(REPO), env=env,
        capture_output=True, text=True,
    )
    
    output = proc.stdout
    print(output[-1500:] if len(output) > 1500 else output)
    if proc.returncode != 0:
        print(f'EVAL FAILED: {proc.stderr[-500:]}')
    
    eval_results.append({
        'name': r['name'],
        'val_loss': r['val_loss'],
        'effective_batch': r['effective_batch'],
        'lr': r['lr'],
        'epochs': r['epochs'],
        'warmup_ratio': r['warmup_ratio'],
        'eval_output': output,
        'returncode': proc.returncode,
        'checkpoint': ckpt,
    })

print(f'\nBenchmark evaluation complete for {len(eval_results)} checkpoints.')

## 9. Sanity Check — Inference with Best Checkpoint

In [ ]:
# Pick the best checkpoint by val_loss for a quick sanity check
best_run = sorted([r for r in results if r['returncode'] == 0 and r['best_checkpoint']],
                  key=lambda x: x['val_loss'])
best_ckpt = best_run[0]['best_checkpoint'] if best_run else None

if best_ckpt:
    prompts = [
        'Explain what a neural network is in one sentence.',
        'Question: Which gas do plants mainly absorb during photosynthesis?\nA) Oxygen\nB) Carbon dioxide\nC) Nitrogen\nD) Helium\nAnswer:',
        'The capital of Switzerland is',
    ]
    for p in prompts:
        print(f'\n{"─"*40}')
        print(f'PROMPT: {p[:80]}...' if len(p) > 80 else f'PROMPT: {p}')
        print(f'{"─"*40}')
        subprocess.run([
            'uv', 'run', 'python', 'main.py',
            '--stage', 'inference',
            '--checkpoint', best_ckpt,
            '--chat-template', 'alpaca',
            '--prompt', p,
            '--max-tokens', '100',
            '--temperature', '0.7',
            '--top-k', '40',
            '--device', 'cuda',
        ], cwd=str(REPO), check=True)
else:
    print('No successful checkpoints to test.')

## 10. Save Results to Drive

In [ ]:
import json, shutil

# Save results JSON (training results + eval results)
results_file = DRIVE_ROOT / f'{RUN_PHASE}_{RUN_STAMP}_results.json'
save_data = {
    'phase': RUN_PHASE,
    'timestamp': RUN_STAMP,
    'training_results': results,
    'eval_results': [{k: v for k, v in er.items() if k != 'eval_output'}
                     for er in eval_results] if eval_results else [],
}
with open(results_file, 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print(f'Results saved to {results_file}')

# Copy ALL checkpoints to Drive (they're small — 40M model ~160MB each)
for r in results:
    if r.get('best_checkpoint') and pathlib.Path(r['best_checkpoint']).exists():
        dest = DRIVE_ROOT / f'{RUN_PHASE}_{r["name"]}_best.pt'
        shutil.copy2(r['best_checkpoint'], dest)
        print(f'Copied {r["name"]} → {dest.name}')

# Copy metrics for each run
for r in results:
    ckpt_dir = pathlib.Path(r['checkpoint_dir'])
    for f_name in ['metrics.csv', 'metrics.jsonl', 'sample_generations.jsonl']:
        src = ckpt_dir / f_name
        if src.exists():
            dest = DRIVE_ROOT / f'{RUN_PHASE}_{r["name"]}_{f_name}'
            shutil.copy2(src, dest)

print(f'\nAll artifacts saved to {DRIVE_ROOT}')